# 6.2 ParaView And VTK Outputs

ParaView output is the best way to inspect meshes, wavefields, material properties, PML regions, source locations, and selected surfaces. This tutorial requests several output targets from one frequency-domain acoustic job, lists the generated files, and renders persistent PyVista screenshots.

By the end, you should be able to request volume, surface, and grid ParaView outputs, discover generated VTU files, and render persistent PyVista screenshots for notebooks.


## How To Read This Tutorial

ParaView output is the spatial audit trail for a run. Traces tell you what receivers measured; VTK/VTU files show the mesh, material properties, PML, sources, fields, and selected surfaces that produced those measurements.

This notebook uses one frequency-domain QC job to request several output targets, then reads the generated files both visually through PyVista screenshots and structurally through field/array inspection.

## Output Target Concepts

ParaView output is job-owned and frequency-domain: the output request belongs to the `FrequencyDomainJob` that produced the VTK files. A time-domain trace job can reconstruct receiver gathers, but ParaView wavefield files should come from an explicit frequency-domain QC job.

| Target | API helper | Use | Typical options |
| --- | --- | --- | --- |
| Volume | `ParaviewOutput.volume(...)` | Full mesh, wavefield, PML, and material-property inspection. | `fields`, `properties`, `sources`, `show_pml`, `order`, `upscale` |
| Surface | `ParaviewOutput.surface(...)` | Interfaces, boundaries, shells, or planes. | `surfaces`, `boundaries`, `shell`, `plane`, `properties` |
| Grid | `ParaviewOutput.grid(...)` | Uniform sampled output for compact visualization or post-processing. | `grid=fs.CartesianGrid(...)`, `fields`, `properties` |

`order` controls the polynomial order used for output interpolation. `upscale` increases output sampling within elements; this is useful for smoother visualization but can make files larger. `parts` can request real, imaginary, and absolute-value fields for complex frequency-domain results. The files are standard VTK/VTU and can be opened in ParaView; the PyVista cells here create persistent screenshots so notebook readers see the result without an interactive viewer.


## Imports

The examples use the public `import frequensolve as fs` API plus standard scientific Python tools for inspection and plotting. Keeping imports ordinary makes the notebook easier to reuse in analysis or operations notebooks.

In [ ]:
from pathlib import Path

import numpy as np
from IPython.display import Image, display
import frequensolve as fs

u = fs.ureg


## Build A Frequency-Domain Model

The model is a simple acoustic two-layer section. The frequency-domain job computes one frequency and writes traces plus ParaView files.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="paraview_outputs",
    path="./scratch/tutorials/paraview_vtk",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="paraview_outputs",
    physics="acoustic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
model.add_surface(name="interface", depth=0.24 * u.km)
model.add_layer(name="basement", properties={"Vp": 2.6 * u.km / u.s, "Rho": 2.3 * u.g / u.cm**3})
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model

sim += model.hex_mesh_generator([8, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim.mesh.set_source_grading(d0=0.02, d1=0.08, factor=2.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

acq = fs.Acquisition()
acq.add_sources(kind="scalar", coords=[[0.5, 0.05]])
hydrophone = fs.ReceiverNode(name="hydrophone")
hydrophone.add_component(name="p", field="pressure")
acq.add_receiver_group(name="surface", device=hydrophone, coords=[[x, 0.04] for x in np.linspace(0.1, 0.9, 41)])
sim += acq
sim += fs.Discretization()

model.plot("vp", figsize=(7, 3), aspect="equal")


## Configure Volume, Surface, And Grid Outputs

The output objects below are plain Python API objects. Inspecting `to_fs(...)` is useful when debugging an exported job JSON or confirming what will be sent to a remote site.


In [ ]:
volume_output = fs.ParaviewOutput.volume(
    name="pv_volume",
    path="pv_volume",
    fields=["pressure"],
    properties=["vp", "rho", "Subdomain"],
    parts=["real", "imag", "abs"],
    sources=[1],
    show_pml=True,
    upscale=1,
    order=2,
)

surface_output = fs.ParaviewOutput.surface(
    name="pv_interface",
    path="pv_interface",
    surfaces=["interface"],
    fields=["pressure"],
    properties=["vp", "Subdomain"],
    parts=["abs"],
    show_pml=False,
    upscale=1,
    order=2,
)

grid = fs.CartesianGrid(n=[121, 61], x0=[0.0, 0.0], x1=[1.0, 0.5], units="km")
grid_output = fs.ParaviewOutput.grid(
    grid,
    name="pv_grid",
    path="pv_grid",
    fields=["pressure"],
    properties=["vp"],
    parts=["abs"],
    sources=[1],
)

[output.to_fs(sim.export_context()) for output in [volume_output, surface_output, grid_output]]


## Run The ParaView Output Job

The run is strict. If ParaView/VTK generation fails on the solver side, the exception should point to the job logs. The plotting cells below assume the run produced `.vtu` files.


In [ ]:
site = fs.Site()
job = fs.FrequencyDomainJob(
    name="freq_paraview",
    simulation=sim,
    f_list=[20.0],
    outputs=[volume_output, surface_output, grid_output],
)
result = site.submit(job).wait()


## Discover Generated Files

Use `result.output_files(...)` rather than constructing paths manually. The base filter matches the output name/path stem and allows solver suffixes such as `_00001`.


In [ ]:
volume_files = result.output_files(base="pv_volume", suffix=".vtu", existing=True)
interface_files = result.output_files(base="pv_interface", suffix=".vtu", existing=True)
grid_files = result.output_files(base="pv_grid", suffix=".vtu", existing=True)
{
    "volume": [str(path) for path in volume_files],
    "interface": [str(path) for path in interface_files],
    "grid": [str(path) for path in grid_files],
}


## Render Persistent Screenshots

These screenshots demonstrate three common review views: material property on the volume mesh, pressure magnitude on the same volume mesh, and the named interface selection. The images are saved under `assets/` so they remain visible after reopening the notebook.


In [ ]:
image_dir = Path("./assets")
image_dir.mkdir(exist_ok=True)

renders = [
    (volume_files[0], "vp", image_dir / "paraview_volume_vp.png"),
    (volume_files[0], "pressure", image_dir / "paraview_volume_pressure_abs.png"),
    (interface_files[0], "vp", image_dir / "paraview_interface_vp.png"),
]
for vtu_file, field, screenshot in renders:
    fs.plot_vtu(
        vtu_file,
        field=field,
        part="abs" if field == "pressure" else "real",
        show_edges=True,
        scalar_bar=True,
        show=False,
        screenshot=screenshot,
        window_size=(1100, 500),
    )
    display(Image(filename=str(screenshot)))


## Load A VTU For Programmatic Inspection

PyVista datasets can also be used for automated checks: inspect point/cell arrays, bounds, or mesh sizes before deciding whether to open the result in ParaView.


In [ ]:
mesh = fs.read_vtu(volume_files[0])
{
    "bounds": mesh.bounds,
    "n_points": mesh.n_points,
    "n_cells": mesh.n_cells,
    "point_arrays": list(mesh.point_data.keys()),
    "cell_arrays": list(mesh.cell_data.keys()),
}


## Before Moving On

Use ParaView output before trusting a large run. Confirm the mesh is in the right place, PML regions exist on the intended boundaries, material properties are present, sources are visible where expected, and requested fields/properties are actually written.

Persistent screenshots belong in release-quality notebooks because they survive kernel restarts and make tutorial results inspectable without an active PyVista session.

## Result Review Checklist

ParaView output should be reviewed as both a visual product and a structured dataset. The notebook screenshots are persistent documentation artifacts, while the VTU metadata checks support automated QA.

| Artifact | What to confirm |
| --- | --- |
| Volume output | Mesh, PML, material properties, source fields, and requested wavefield parts are present. |
| Surface output | Named surfaces or slices produce smaller focused datasets for interface review. |
| Saved screenshots | Images remain visible after reopening the notebook without a live PyVista session. |
| `read_vtu(...)` metadata | Bounds, point/cell counts, and available arrays match the requested output options. |

Use ParaView for large interactive exploration. Use PyVista screenshots in release notebooks so readers can see evidence of the output even when they are not running a live visualization kernel.
